# 코랩 A100 · 수혁 1024px 모델 test 추론

수혁 님의 **B3_px1024** (dev 0.6616) 는 5060 Ti 16GB 에서 1024px 가 메모리를 넘겨 문항당 26.8초가 걸렸습니다.
test 6,714문항이면 50시간이라 PC 로는 불가능해서, **A100(40GB)** 에서 추론만 합니다.

| | PC (16GB) | A100 (40GB) |
|---|---|---|
| 문항당 | 26.8초 | 약 0.8초 (예상) |
| test 전체 | 50시간 | **약 90분 · 30유닛** |

### 준비
1. 드라이브에 **`B3_px1024.zip`** 이 있으면 됩니다 (이름·위치가 다르면 CONFIG 의 `ADAPTER_ZIP` 만 수정)
2. 노트북이 zip 을 코랩 로컬에 풀고 `adapter_config.json` 을 찾아 알아서 경로를 잡습니다
3. 런타임 **A100**, `MyDrive/data.zip` 확인
4. 결과(제출 csv · test 확률)는 **드라이브** `challenge/output/` 에 저장됩니다

### 안전장치
본 추론(90분) 전에 **dev 100문항으로 재현 확인**을 합니다. 수혁 님 PC 에서 나온 답과 90% 이상 일치해야 진행하고,
아니면 멈춥니다 (해상도·모델을 잘못 불러온 채로 유닛을 쓰지 않도록).

In [ ]:
CONFIG = {
    "RUN_NAME":    "B3_px1024_수혁",
    "ADAPTER_ZIP": "/content/drive/MyDrive/B3_px1024.zip",   # 드라이브에 올린 zip
    "BASE_REPO":   "Qwen/Qwen2.5-VL-3B-Instruct",   # B3 어댑터의 기본 모델 (adapter_config.json 으로 확인)
    "IMAGE_SIZE":  1024,                          # 학습 때와 반드시 같아야 함
    "LOAD_IN_4BIT": True,                         # 학습 때와 같게 (4bit)
    "SPLIT_SEED":  42,
    "DRIVE_DIR":   "/content/drive/MyDrive/challenge",
    "DATA_ARCHIVE": "/content/drive/MyDrive/data.zip",
}
print(CONFIG["RUN_NAME"], "|", CONFIG["IMAGE_SIZE"], "px")

# 1. 드라이브 · 데이터 · 어댑터 확인

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path(CONFIG["DRIVE_DIR"]); DRIVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = DRIVE_DIR / "output"; SUB_DIR = OUTPUT_DIR / "submissions"
SUB_DIR.mkdir(parents=True, exist_ok=True)

# --- 어댑터 zip 을 코랩 로컬에 해제 (폴더가 한 겹 더 있어도 찾아냄) ---
zp = Path(CONFIG["ADAPTER_ZIP"])
assert zp.exists(), f"zip 을 찾지 못했습니다: {zp}\n드라이브에서 파일 이름과 위치를 확인하세요"
dest = Path("/content/adapter"); dest.mkdir(exist_ok=True)
if not list(dest.rglob("adapter_config.json")):
    with zipfile.ZipFile(zp) as z:
        z.extractall(dest)
hits = list(dest.rglob("adapter_config.json"))
assert hits, "zip 안에서 adapter_config.json 을 찾지 못했습니다 (폴더 통째로 압축했는지 확인)"
ADAPTER_DIR = hits[0].parent
files = sorted(f.name for f in ADAPTER_DIR.iterdir())
assert "adapter_model.safetensors" in files, f"adapter_model.safetensors 가 없습니다: {files}"
print("어댑터:", ADAPTER_DIR)
print("  파일:", files)
print("  dev_probs.npy:", "있음 (재현 확인 가능)" if "dev_probs.npy" in files else "없음 (재현 확인 생략됨)")

# --- 데이터 zip 해제 ---
WORK = Path("/content/challenge"); WORK.mkdir(exist_ok=True)
if not list(WORK.rglob("test.csv")):
    subprocess.run(["unzip", "-q", CONFIG["DATA_ARCHIVE"], "-d", str(WORK)], check=True)
DATA_DIR = next(WORK.rglob("test.csv")).parent
print("데이터:", DATA_DIR)

# 2. 설치 (Unsloth 아님 — 수혁 님이 쓴 HF 방식 그대로)

In [ ]:
!pip install -q "transformers>=4.57.0,<5.0.0" "peft>=0.17" "bitsandbytes>=0.45" accelerate

# 3. 불러오기 · A100 확인

In [ ]:
import torch, numpy as np, pandas as pd, time
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
Image.MAX_IMAGE_PIXELS = None
import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
gpu = torch.cuda.get_device_name(); vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} | {vram:.0f}GB")
assert vram > 30, "A100 이 아닙니다 (1024px 는 40GB 급이 필요). 런타임 유형을 바꾸세요"

# 4. 데이터 · dev 평가셋 복원

In [ ]:
test_df    = pd.read_csv(DATA_DIR / "test.csv")
dev_df     = pd.read_csv(DATA_DIR / "dev.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
def image_path(rel):
    return DATA_DIR / str(rel).replace("\\", "/")
assert image_path(test_df["path"].iloc[0]).exists()

# 수혁 님 노트북과 같은 방식으로 dev 평가셋 복원 (다수결 · 동률 제외)
VOTE_COLS = [f"answer{i}" for i in range(1, 6)]
def to_letter(v, row):
    if pd.isna(v): return None
    s = str(v).strip().lower()
    if s in list("abcd"): return s
    for L in "abcd":
        if str(row[L]).strip().lower() == s: return L
    return None
def majority(votes):
    top = Counter(v for v in votes if v).most_common()
    if not top or (len(top) > 1 and top[0][1] == top[1][1]): return None
    return top[0][0]
d = dev_df.copy()
d["votes"]  = d.apply(lambda r: [to_letter(r[c], r) for c in VOTE_COLS], axis=1)
d["answer"] = d["votes"].apply(majority)
order_plain = d[d["answer"].notna()].reset_index(drop=True)
order_shuf  = order_plain.sample(n=len(order_plain), random_state=CONFIG["SPLIT_SEED"]).reset_index(drop=True)
print("test", len(test_df), "| dev 평가셋", len(order_plain))

# 5. 기본 모델 + 수혁 님 어댑터

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM
from peft import PeftModel

PIX = CONFIG["IMAGE_SIZE"] ** 2
try:
    processor = AutoProcessor.from_pretrained(CONFIG["BASE_REPO"], min_pixels=PIX, max_pixels=PIX)
except TypeError:
    processor = AutoProcessor.from_pretrained(CONFIG["BASE_REPO"])
tok = processor.tokenizer
tok.padding_side = "right"
print("min/max_pixels:", getattr(processor.image_processor, "min_pixels", None),
                         getattr(processor.image_processor, "max_pixels", None))

kw = {}
if CONFIG["LOAD_IN_4BIT"]:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
base = AutoVLM.from_pretrained(CONFIG["BASE_REPO"], device_map={"": 0},
                               attn_implementation="eager", torch_dtype=torch.bfloat16, **kw)
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
print(f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f}GB")

# 6. 채점 함수 · 재현 확인 (90% 미만이면 중단)

In [ ]:
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")
def build_mc_prompt(q, a, b, c, dd):
    return (f"{q}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {dd}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")
LETTERS = list("abcd")
LETTER_IDS = []
for ch in LETTERS:
    ids = tok.encode(ch, add_special_tokens=False); assert len(ids) == 1, ch
    LETTER_IDS.append(ids[0])

@torch.no_grad()
def predict_letters(df, desc):
    preds, probs = [], []
    for i in tqdm(range(len(df)), desc=desc, unit="it"):
        row = df.iloc[i]
        img = Image.open(image_path(row["path"])).convert("RGB")
        msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [{"type": "image", "image": img},
                 {"type": "text", "text": build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to("cuda")
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(**inputs).logits[:, -1, :].float()
        sel = logits[0, LETTER_IDS]
        preds.append(LETTERS[int(sel.argmax())])
        probs.append(torch.softmax(sel, dim=-1).cpu().tolist())
    return preds, probs

# --- 재현 확인 : 수혁 님 PC 결과와 같은 답을 내는지 ---
K = 100
pfile = ADAPTER_DIR / "dev_probs.npy"
if not pfile.exists():
    print("dev_probs.npy 가 없어 재현 확인 생략 (속도만 측정)")
    t0 = time.time(); predict_letters(test_df.head(20), "속도 측정"); sec = (time.time() - t0) / 20
else:
    saved = [LETTERS[i] for i in np.load(pfile).argmax(1)]
    assert len(saved) == len(order_plain), f"저장된 dev 답 {len(saved)} != {len(order_plain)}"
    best = 0.0
    for name, df_ in [("섞인 순서", order_shuf), ("원래 순서", order_plain)]:
        t0 = time.time(); p, _ = predict_letters(df_.head(K), f"재현 확인 ({name})"); sec = (time.time() - t0) / K
        agree = float(np.mean([x == y for x, y in zip(p, saved[:K])]))
        print(f"  {name}: 일치율 {agree:.0%} | 문항당 {sec:.2f}초")
        best = max(best, agree)
        if agree >= 0.9: break
    assert best >= 0.9, "수혁 님 PC 결과와 다르게 답합니다 (IMAGE_SIZE / LOAD_IN_4BIT / BASE_REPO 확인)"
print(f"-> test 6,714문항 예상 {sec * len(test_df) / 60:.0f}분 ≈ {sec * len(test_df) / 3600 * 18:.0f}유닛")

# 7. test 추론 · 제출 파일

In [ ]:
t0 = time.time()
test_preds, test_probs = predict_letters(test_df, f"test [{CONFIG['RUN_NAME']}]")
ID_COL, ANS_COL = sample_sub.columns[0], sample_sub.columns[1]
sub = sample_sub.copy()
sub[ANS_COL] = sub[ID_COL].map(dict(zip(test_df[ID_COL], test_preds)))
assert sub[ANS_COL].isna().sum() == 0
sub_path = SUB_DIR / f"submission_{CONFIG['RUN_NAME']}.csv"
sub.to_csv(sub_path, index=False)
np.save(OUTPUT_DIR / f"test_probs_{CONFIG['RUN_NAME']}.npy", np.array(test_probs))   # 드라이브에 저장 (앙상블 재료)
print(f"저장: {sub_path} ({(time.time() - t0) / 60:.1f}분) | 분포 {sub[ANS_COL].value_counts().to_dict()}")
print(f"캐글 설명란: {CONFIG['RUN_NAME']} | 1024px · dev 0.6616")